In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
model=ChatOpenAI()

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# Load web data
loader = WebBaseLoader("https://www.langchain.dev/langgraph/")
data = loader.load()

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.create_documents([data[0].page_content])
documents

[Document(metadata={}, page_content='LangChain: Observe, Evaluate, and Deploy Reliable AI Agents\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJoin us May 13th &\xa0May 14th at Interrupt, the Agent Conference by LangChainBuy tickets\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceDeploymentShip and scale agents in productionAgent BuilderNo-code agents for real workOpen Source FrameworksdeepagentsBuild long-running agents for complex taskslangchainQuick start agents with any model providerlanggraphBuild reliable agents with low-level controlLearn\n\nResourcesBlogCustomer StoriesGuidesHow-ToLangChain AcademyYouTubeDocumentationCommunityLangSmith for StartupsEventsCommunityDocsCompany\n\nAboutCareersPartnersPricingTry LangSmith\n\nGet a demo\n\nTry LangSmith\n\nGet a demo\n\nShip agents that work wowObserve, evaluate, and deploy agents with LangSmith, the agent

In [12]:
documents = text_splitter.split_documents(data)
documents

[Document(metadata={'source': 'https://www.langchain.dev/langgraph/', 'title': 'LangChain: Observe, Evaluate, and Deploy Reliable AI Agents', 'description': 'LangChain provides the engineering platform and open source frameworks developers use to build, test, and deploy reliable AI agents.', 'language': 'en'}, page_content='LangChain: Observe, Evaluate, and Deploy Reliable AI Agents\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJoin us May 13th &\xa0May 14th at Interrupt, the Agent Conference by LangChainBuy tickets\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceDeploymentShip and scale agents in productionAgent BuilderNo-code agents for real workOpen Source FrameworksdeepagentsBuild long-running agents for complex taskslangchainQuick start agents with any model providerlanggraphBuild reliable agents with low-level controlLearn\n\nResourcesBlogCustomer Stor

In [13]:
from langchain_community.vectorstores import FAISS
vector = FAISS.from_documents(documents, OpenAIEmbeddings())
 

In [14]:
retriever = vector.as_retriever()

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="chat_history"),
        ("user","{input}"),
        ("user", "Given the above conversation, generate a search \
        query to look up in order to get information relevant to the conversation.")
    ])

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.vectorstores import Chroma

# -----------------------------
# 1️⃣ LLM initialization and Embeddings
# -----------------------------

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()

# -----------------------------
# 2️⃣ documents
# -----------------------------

docs = documents

# -----------------------------
# 3️⃣ Chroma Vector Store
# -----------------------------

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings
)

retriever = vector_store.as_retriever()

# -----------------------------
# 4️⃣ History-aware Question Prompt
# -----------------------------

question_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the conversation history and a follow-up question, "
               "rewrite the follow-up to be a standalone search query."),
    ("human", "Chat History:\n{chat_history}\n\nFollow-up Question:\n{question}")
])

# -----------------------------
# 5️⃣ Chat history formatter
# -----------------------------

def format_history(history):
    return "\n".join(
        f"Human: {m.content}" if isinstance(m, HumanMessage)
        else f"AI: {m.content}"
        for m in history
    )

# -----------------------------
# 6️⃣ RAG Fuction
# -----------------------------

def ask_rag(history, user_question):

    # 1️⃣ Standalone query 
    hist_str = format_history(history)

    standalone_prompt = question_prompt.invoke({
        "chat_history": hist_str,
        "question": user_question
    })

    standalone_query = llm.invoke(standalone_prompt).content

    # 2️⃣ Retrieve documents
    retrieved_docs = retriever.invoke(standalone_query)

    # 3️⃣ Final Answer Prompt
    answer_prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer the question using ONLY the documents below:\n\n{docs}"),
        ("human", "{question}")
    ])

    final_prompt = answer_prompt.invoke({
        "docs": "\n\n".join(doc.page_content for doc in retrieved_docs),
        "question": user_question
    })

    response = llm.invoke(final_prompt).content

    return response

# -----------------------------
# 7️⃣  Test RAG pipeline
# -----------------------------

history = [
    HumanMessage(content="Can I use LangGraph for agent runtimes?"),
    AIMessage(content="Yes!")
]

answer = ask_rag(history, "Tell me how")
print(answer)

The documents provided outline two main processes: Evaluation and Deployment.

**Evaluation** involves:
- Using real-world usage for iterative improvement.
- Capturing production traces and turning them into test cases.
- Scoring agents through a combination of human review and automated evaluations.
- Implementing reusable LLM-as-judge and multi-turn evaluations.
- Calibrating evaluations with human feedback and annotations.
- Utilizing both online and offline scoring methods.

**Deployment** focuses on:
- Shipping and scaling agents in production, which operate for long durations and require asynchronous collaboration with humans and other agents.
- Providing an agent server that includes memory, conversational threads, and durable checkpointing.
- Ensuring the infrastructure is fault-tolerant and scalable to handle any workload.
- Supporting human-in-the-loop interactions, input concurrency, and background agents.
- Offering type-safe streaming of messages, UI components, and custom